# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook demonstrates loading and exploring the Kilifi County FAIR² mental health survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema and is published at:

https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

* Load and inspect metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the mlcroissant Dataset object
dataset = mlc.Dataset(url)

# Access the metadata object (note: do not treat as dict/list)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}\nVersion: {metadata.version}\nIdentifier: {metadata.identifier}")

## 2. Data Overview

* Review available record sets, fields, and their IDs.

For Croissant datasets, each record set, field, and column is referenced by its `@id`.

Let's enumerate available record sets and their fields.

In [ ]:
# List record sets and fields, referencing each by @id
record_sets = list(dataset.record_sets())
record_set_ids = []
for rs in record_sets:
    rs_id = rs['@id']
    record_set_ids.append(rs_id)
    print(f"RecordSet @id: {rs_id}")
    if 'field' in rs:
        print("  Fields:")
        for field in rs['field']:
            print(f"    Field @id: {field['@id']}")
            if 'column' in field:
                for column in field['column']:
                    print(f"      Column @id: {column['@id']}")
    print()

## 3. Data Extraction

* Load records from a chosen record set (using its `@id`) into a DataFrame for analysis.

In [ ]:
# Collect all available record set IDs
# (record_set_ids variable already created above)
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded RecordSet {rs_id} with shape: {df.shape}")

# For demonstration, select the first record set
main_record_set_id = record_set_ids[0] if record_set_ids else None

if main_record_set_id:
    print(f"Columns for RecordSet {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

* Filter, normalize, and group data using column `@id`. Remove outliers, transform distributions, or categorize data by attributes.

Below, we select a numeric field (such as GAD-7 score) by its column `@id`.

In [ ]:
# Identify a numeric field, e.g., GAD-7 score
df = dataframes[main_record_set_id]

# You might need to inspect columns to find the GAD-7 field @id
# For demonstration, suppose GAD-7 score column is 'http://mlcommons.org/croissant/GAD7Score'
# Update the field below to match your column @id!
numeric_field_id = None
for col in df.columns:
    if 'GAD7' in col or 'GAD-7' in col:
        numeric_field_id = col
        break
if not numeric_field_id:
    numeric_field_id = df.columns[0]  # fallback

print(f"Using numeric column @id: {numeric_field_id}")

# Filter records above a threshold
threshold = 10
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by an attribute e.g. 'Gender' field @id
    group_field_id = None
    for col in df.columns:
        if 'Gender' in col or 'gender' in col or 'sex' in col:
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df.head())
else:
    print(f"Column {numeric_field_id} is not numeric or contains missing values.")

## 5. Visualization

* Visualize the distribution of GAD-7 scores and their grouping by gender using column `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.tight_layout()
    plt.show()

    if group_field_id:
        plt.figure(figsize=(7,4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

* In this notebook, we loaded and explored the FAIR² Kilifi County mental health survey dataset using `mlcroissant`, referencing all data elements by their Croissant `@id`. We demonstrated filtering, normalization, grouping, and visualizations of key attributes (such as GAD-7 scores) using standardized schema identifiers, readying the data for further statistical and AI analysis.